# 🏠 House Price Prediction — Step-by-Step Notebook

This notebook walks through the full ML pipeline for house price prediction.

**Sections:**
1. Dataset Generation
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training & Comparison
5. Evaluation & Insights
6. Price Prediction

## Step 1 — Setup & Dataset Generation

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from generate_dataset import generate_housing_data

# Generate dataset
df = generate_housing_data(n=1500, seed=42)
df.to_csv('../data/houses.csv', index=False)

print(f'Dataset shape: {df.shape}')
df.head()

## Step 2 — EDA: Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['SalePrice']/1e6, bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('Sale Price Distribution')
axes[0].set_xlabel('Price (₹ Millions)')
axes[1].hist(np.log1p(df['SalePrice']), bins=40, color='#DD8452', edgecolor='white')
axes[1].set_title('Log-Price Distribution')
axes[1].set_xlabel('log(1 + Price)')
plt.tight_layout()
plt.show()

print(df['SalePrice'].describe())

## Step 3 — Correlation Heatmap

In [ ]:
from features import add_features
df = add_features(df)

num_cols = ['SalePrice','GrLivArea','TotalBsmtSF','LotArea',
            'OverallQual','GarageCars','FullBath','Age','TotalSF']
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Step 4 — Feature Engineering & Preprocessing

In [ ]:
from features import get_feature_lists
from pipeline import build_preprocessor
from sklearn.model_selection import train_test_split

NUM, CAT = get_feature_lists()
X = df[NUM + CAT]
y = df['SalePrice']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {Xtr.shape}, Test: {Xte.shape}')
print('Engineered features:', [f for f in NUM if f not in df.columns[:20]])

## Step 5 — Train & Compare Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'Linear':          Pipeline([('pre', build_preprocessor()), ('m', LinearRegression())]),
    'Ridge':           Pipeline([('pre', build_preprocessor()), ('m', Ridge(alpha=10))]),
    'Decision Tree':   Pipeline([('pre', build_preprocessor()), ('m', DecisionTreeRegressor(max_depth=8, random_state=42))]),
    'Random Forest':   Pipeline([('pre', build_preprocessor()), ('m', RandomForestRegressor(n_estimators=200, random_state=42))]),
    'Gradient Boost':  Pipeline([('pre', build_preprocessor()), ('m', GradientBoostingRegressor(n_estimators=200, random_state=42))]),
}

print(f'{"Model":<20} {"MAE":>12} {"RMSE":>12} {"R²":>8}')
print('-'*56)
for name, pipe in models.items():
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    mae  = mean_absolute_error(yte, pred)
    rmse = np.sqrt(mean_squared_error(yte, pred))
    r2   = r2_score(yte, pred)
    print(f'{name:<20} ₹{mae:>10,.0f}   ₹{rmse:>10,.0f}   {r2:>6.4f}')

## Step 6 — Actual vs Predicted Chart

In [ ]:
best_pipe = models['Random Forest']
pred = best_pipe.predict(Xte)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(yte/1e6, pred/1e6, alpha=0.35, s=15)
lims = [min(yte.min(),pred.min())/1e6, max(yte.max(),pred.max())/1e6]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual (₹M)')
axes[0].set_ylabel('Predicted (₹M)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

residuals = yte - pred
axes[1].hist(residuals/1e6, bins=40, color='#55a868', edgecolor='white')
axes[1].axvline(0, color='red', lw=2, ls='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (₹M)')

plt.tight_layout()
plt.show()

## Step 7 — Predict New House Price

In [ ]:
new_house = {
    'MSZoning': 'RL', 'LotArea': 7500, 'Neighborhood': 'Koramangala',
    'BldgType': '1Fam', 'HouseStyle': '2Story', 'OverallQual': 8,
    'OverallCond': 6, 'YearBuilt': 2010, 'YearRemodAdd': 2018,
    'Foundation': 'PConc', 'TotalBsmtSF': 1000.0, '1stFlrSF': 1000,
    '2ndFlrSF': 800, 'GrLivArea': 1800, 'FullBath': 2, 'HalfBath': 1,
    'BedroomAbvGr': 3, 'TotRmsAbvGrd': 8, 'Fireplaces': 1,
    'GarageCars': 2.0, 'GarageArea': 400.0,
    'PavedDrive': 'Y', 'CentralAir': 'Y', 'KitchenQual': 'Gd',
}

inp = add_features(pd.DataFrame([new_house]))
price = best_pipe.predict(inp[NUM + CAT])[0]

print(f'🏡 Predicted Price: ₹{price:,.0f}')
print(f'   Range (±8%):    ₹{price*0.92:,.0f} – ₹{price*1.08:,.0f}')